# Lyric Semantic Search — searching songs by meaning, not just words

**YDL 2026 · Applied NLP · Day 2 Lab Project**

We take ~57k English songs, split every song's lyrics into single lines, and build **three** search
indexes over those lines:

| Method | What it compares | Technique | Strong at |
|--------|------------------|-----------|-----------|
| **tf-idf** | matching **words** | tf-idf + cosine | exact words |
| **GloVe** | the **meaning** of a line | averaged embeddings + cosine | synonyms, mood |
| **trigram** | **3-character chunks** (lon·ond·ndo·don) | char-3gram + cosine | typos, fuzzy |

> No heavy models (no BERT/Gemma) — only Day-2 tools: tf-idf, averaged word embeddings,
> cosine similarity, and character n-grams.

**The core idea:** the query *"feeling heartbroken and alone"* matched by keywords only finds lines that
literally contain "feeling"; matched by meaning it finds genuinely sad lines like *"Feeling sad and all
alone"* — even with no shared words.

**Note on the corpus:** we search the **lyric text itself** (one line per document). The artist and song
title are only labels showing where each line came from — we do **not** search by title or author.

## 0. Load the indexes (tf-idf + GloVe + trigram)

In [1]:
import search   # search_tfidf / search_glove / search_trigram / compare
search._load()
print(f"lines in corpus:   {len(search._lines):,}")
print(f"tf-idf vocab:      {len(search._tfidf.vocabulary_):,} words")
print(f"GloVe matrix:      {search._gmat.shape[0]:,} lines x {search._gmat.shape[1]}d")
print(f"trigram vocab:     {len(search._tri.vocabulary_):,} character 3-grams")

lines in corpus:   180,483
tf-idf vocab:      15,398 words
GloVe matrix:      180,483 lines x 100d
trigram vocab:     10,049 character 3-grams


## 1. Three modes on the same query

The queries are written in **our own words** — no line contains them verbatim.
`compare()` runs all three methods side by side.

In [2]:
search.compare("feeling heartbroken and alone")


Query: "feeling heartbroken and alone"

  ── tf-idf   (keywords) ──
     1.00  "There's a feeling I have"  — Madonna, Like A Flower
     1.00  "Can you see what I'm feeling?"  — Wilson Phillips, The Dream Is Still Alive
     1.00  "There's a feeling you give me"  — Willie Nelson, Be There For You
     1.00  "More than a feeling"  — Westlife, Lighthouse
     1.00  "I get a feeling in me"  — Alison Krauss, Oh, Atlanta

  ── GloVe    (meaning)  ──
     0.95  "Feeling sad and all alone"  — Britney Spears, Luv The Hurt Away
     0.94  "When I'm feeling scared and lonely"  — Kylie Minogue, Count The Days
     0.93  "You're feeling all alone"  — Glee, I'll Stand By You
     0.93  "And when I feel lonely"  — John Mellencamp, The Breakout
     0.93  "Why you're feeling so alone"  — Free, Catch A Train

  ── trigram  (fuzzy)    ──
     0.59  "Heartbroken when you left my world"  — Usher, Throwback
     0.56  "Broken down and all alone"  — Hillsong, Glow
     0.53  "Tonight I'm all alone and bro

In [3]:
search.compare("a cold and lonely winter night")


Query: "a cold and lonely winter night"

  ── tf-idf   (keywords) ──
     0.86  "On a cold winter night"  — Harry Belafonte, Glory Manger
     0.86  "On a cold winter's night"  — Olivia Newton-John, The First Noel
     0.81  "A cold and lonely night"  — Phish, Icculus
     0.77  "To keep you cold as winter"  — Eurythmics, Cool Blue
     0.74  "On a cold winter's night that was so deep"  — Weezer, The First Noel

  ── GloVe    (meaning)  ──
     0.98  "A cold and lonely night"  — Phish, Icculus
     0.97  "On a cold winter night"  — Harry Belafonte, Glory Manger
     0.97  "The cold and lonely nights,"  — Roxette, Like Lovers Do
     0.95  "A winter that's gray and cold"  — Billie Holiday, Body And Soul
     0.95  "Little darling, it's been a long cold lonely winter"  — Demi Lovato, Here Comes The Sun

  ── trigram  (fuzzy)    ──


     0.85  "A cold and lonely night"  — Phish, Icculus
     0.78  "On a cold winter night"  — Harry Belafonte, Glory Manger
     0.70  "The cold and lonely nights,"  — Roxette, Like Lovers Do
     0.67  "When nights are cold and lonely"  — Patsy Cline, Stand By Your Man
     0.67  "On a cold winter's night"  — Olivia Newton-John, The First Noel


### What about typos?

This is where trigrams shine. The misspelled query **"heartbrokn and lonley"**:
tf-idf finds nothing (no exact words), GloVe breaks (the misspelled words aren't in its vocabulary),
but **trigram still finds** the right lines through shared 3-character chunks.

In [4]:
search.compare("heartbrokn and lonley")   # intentional typos


Query: "heartbrokn and lonley"

  ── tf-idf   (keywords) ──
     (no keyword overlap — nothing found)

  ── GloVe    (meaning)  ──
     1.00  "And moaner moaner moaner moaner"  — Underworld, Moaner
     1.00  "1 and 2 and 3 and 4"  — P!nk, Respect
     1.00  "We're floatin' 'round and 'round"  — Jimi Hendrix, You've Got Me Floating
     1.00  "I'm Rockin' and rollin'"  — Deep Purple, A Castle Full Of Rascals
     1.00  "And 'round and 'round"  — Randy Travis, Everything And All

  ── trigram  (fuzzy)    ──
     0.44  "Heartbreak, heartbreak here I come."  — Nina Simone, Can't Get Out Of This Mood
     0.43  "Heartbroken when you left my world"  — Usher, Throwback
     0.41  "Are you a heartbreaker"  — P!nk, Heartbreaker
     0.39  "Heartbreak here I come."  — Nina Simone, Can't Get Out Of This Mood
     0.38  "And those heartbreaking things"  — David Bowie, I Keep Forgetting


**Summary — each method wins in its own regime, so they complement each other:**
- **tf-idf** — exact words (names, places), but blind to synonyms and typos;
- **GloVe** — meaning and mood (sad ≈ heartbroken ≈ lonely), but unaware of typos;
- **trigram** — robust to typos and fuzzy spelling, but understands no meaning.

That is why the artifact ships a **mode switcher**, not one single "correct" search.

## 2. Interactive search (3-mode switcher)

A standalone search page (`artifacts/search.html`) with **Meaning / Keywords / Fuzzy** buttons —
switch the mode and compare. Everything runs in the browser (JS): GloVe averaging + cosine,
tf-idf over words, and Dice over character trigrams. One file, no server, no heavy models.

> Open `artifacts/search.html` in a browser, or use it right here:

In [5]:
from IPython.display import IFrame
IFrame("artifacts/search.html", width="100%", height=640)